# D365 F&O Customer Cleansing — Double7 (`dou7`) · Adept (`bnn`)

**เป้าหมาย:** ลูกค้า D365 ของ Double7 / Adept เป็น **B2B** → ทำเป็น *ชุดนิติบุคคล* ไม่เข้ากระบวนการจับคู่บุคคล
นับว่ามีบริษัทจริงกี่ราย · บัญชีซ้ำกันไหม (ภายในบริษัทและระหว่าง Double7 ↔ Adept) · ยอดขายอยู่กับลูกค้าประเภทไหน

| ขั้น | ทำอะไร |
|---|---|
| Extract | `tm_Customer` (บัญชี) + สรุปจาก `tb_SalesInvoice` ต่อบัญชี (ชื่อ · จำนวนบรรทัด · ยอด · วันแรก/ล่าสุด) |
| Name | `tm_Customer.CustomerName` ถูก mask → ใช้ชื่อจากใบขาย (`CustomerName` ตาม `CustomerAccount`) |
| Classify | บริษัท · ร้าน/ลูกค้าเงินสด · ชื่อบุคคล · ไม่มีชื่อ (ไม่มียอดขาย) |
| Company key | ตัด บริษัท / จำกัด / (มหาชน) / สาขา / Co., Ltd. / วรรคตอน → คีย์เทียบชื่อ |
| Dedup | บัญชีที่คีย์ชื่อบริษัทเดียวกัน = บริษัทเดียวกัน · เทียบข้าม Double7 ↔ Adept |

**ความปลอดภัย:** read-only (`fo_raw` · `syndpdev001`) · ชื่อบุคคลที่ปนอยู่ mask ทุกตัวอย่าง · ผลรันอยู่ใน `scripts/fo/_out/`

In [ ]:
import sys, json, time, collections, os, re, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, r'C:\Projects\my-first-project\scripts')
import db
from ssot.cleanse import *
pd.set_option('future.infer_string', False)
OUT = r'C:\Projects\my-first-project\scripts\fo\_out'
COS = ('dou7', 'bnn'); CO_NAME = {'dou7': 'Double7', 'bnn': 'Adept'}
cn = db.connect('fo_raw', timeout=900); cur = cn.cursor()
print('connected')

## Extract

In [ ]:
t0 = time.time()
cur.execute("SELECT SysCompanyId, CustomerAccount, CustomerGroup, TaxHeadoffice, SysCreatedDateTime FROM dbo.tm_Customer WITH (NOLOCK) WHERE SysCompanyId IN (?, ?)", *COS)
acct = pd.DataFrame.from_records(cur.fetchall(), columns=['co', 'acct', 'grp', 'ho', 'created'])
cur.execute("SELECT SysCompanyId, CustomerAccount, CustomerName, COUNT(*), SUM(LineAmount), MIN(InvoiceDate), MAX(InvoiceDate) "
            "FROM dbo.tb_SalesInvoice WITH (NOLOCK) WHERE SysCompanyId IN (?, ?) GROUP BY SysCompanyId, CustomerAccount, CustomerName", *COS)
inv = pd.DataFrame.from_records(cur.fetchall(), columns=['co', 'acct', 'name', 'lines', 'amount', 'first', 'last'])
inv['amount'] = inv.amount.astype(float)
print(f'tm_Customer {len(acct):,} accounts · invoice (account, name) pairs {len(inv):,} · {time.time() - t0:,.0f}s')
display(acct.groupby('co').size().rename('accounts').to_frame().join(inv.groupby('co').agg(accounts_with_sales=('acct', 'nunique'), lines=('lines', 'sum'), amount=('amount', 'sum'))))

## ชื่อต่อบัญชี
1 บัญชีอาจมีหลายชื่อในใบขาย (แก้ชื่อ / ใส่สาขา) → ใช้ชื่อที่มีบรรทัดมากสุดเป็นชื่อหลัก · บัญชีที่ไม่มียอดขายจะไม่มีชื่อ (ชื่อใน `tm_Customer` ถูก mask)

In [ ]:
inv['name'] = inv.name.fillna('').astype(str).str.strip()
names_per_acct = inv.groupby(['co', 'acct']).name.nunique()
main = inv.sort_values('lines', ascending=False).drop_duplicates(['co', 'acct'])[['co', 'acct', 'name']]
tot = inv.groupby(['co', 'acct']).agg(lines=('lines', 'sum'), amount=('amount', 'sum'), first=('first', 'min'), last=('last', 'max')).reset_index()
A = acct.merge(main, on=['co', 'acct'], how='left').merge(tot, on=['co', 'acct'], how='left')
A['name'] = A.name.fillna(''); A[['lines', 'amount']] = A[['lines', 'amount']].fillna(0)
print(f"บัญชีที่มีชื่อในใบขาย > 1 ชื่อ: {int((names_per_acct > 1).sum()):,}")
print(f"บัญชีในใบขายที่ไม่มีใน tm_Customer: {len(tot.merge(acct, on=['co', 'acct'], how='left', indicator=True).query('_merge == \"left_only\"')):,}")

## Classify
- **intercompany** = บริษัทในเครือ (คอมเซเว่น · BaNANA · Studio7 · Double7 · Adept) → ยอดขายระหว่างกัน ไม่ใช่ลูกค้า
- **company** = มีคำนิติบุคคล (บริษัท · หจก. · จำกัด · Co., Ltd. · โรงเรียน · ร้าน …) หรือเป็นสาขาของร้านตัวแทน (`… สาขา 1`)
- **cash / general** = ลูกค้าเงินสด / ลูกค้าทั่วไป
- **person-like** = ไม่มีคำนิติ และผ่านกฎชื่อคน (อาจเป็นบุคคลจริง หรือบริษัทที่ไม่ได้เขียนคำนำหน้า → ต้องให้ทีมธุรกิจดู)
- **no name** = ไม่มียอดขาย จึงไม่มีชื่อ

In [ ]:
nm = A.name
GROUP = INTERNAL + r'|ดับเบิ้ล\s?เซเว่น|double\s?7|dou7|อะเด็พท์|อะเดปท์|อะเด็ปท์|adept'
is_ic = nm.str.contains(GROUP, case=False)
is_co = (nm.str.contains(JURISTIC, case=False) | nm.str.contains(r'สาขา|branch', case=False)) & ~is_ic
is_store = nm.str.contains(r'เงินสด|cash|ทั่วไป|general|walk[\s-]?in', case=False) & ~is_co & ~is_ic
cl = clean_names(nm, pd.Series('', index=nm.index))
is_person = ~is_co & ~is_store & ~is_ic & (nm != '') & (cl['junk'] == '').to_numpy()
A['kind'] = np.select([nm == '', is_ic, is_co, is_store, is_person], ['no name', 'intercompany', 'company', 'cash / general', 'person-like'], 'other')
K = A.groupby(['co', 'kind']).agg(accounts=('acct', 'size'), lines=('lines', 'sum'), amount=('amount', 'sum'))
K['amount_%'] = K.amount / K.groupby(level=0).amount.transform('sum') * 100
display(K.style.format({'accounts': '{:,.0f}', 'lines': '{:,.0f}', 'amount': '{:,.0f}', 'amount_%': '{:.1f}'}))
ex = []
for k in ['intercompany', 'company', 'cash / general', 'person-like', 'other']:
    for v in A[A.kind == k].name.head(5):
        ex.append((k, mask_org(v) if k != 'person-like' else mask_tok(v)))
pd.DataFrame(ex, columns=['kind', 'name (masked)'])

## Company key + dedup
คีย์ = ชื่อหลังตัดคำประเภทนิติ / สาขา / วรรคตอน · บัญชีที่คีย์เดียวกัน = บริษัทเดียวกัน (เช่น เปิดบัญชีแยกสาขา หรือเปิดทั้งที่ Double7 และ Adept)

In [ ]:
C2 = A[A.kind == 'company'].copy()
C2['key'] = company_key(C2.name)
C2 = C2[C2.key.str.len() >= 2]
g = C2.groupby('key').agg(accounts=('acct', 'size'), cos=('co', 'nunique'), lines=('lines', 'sum'), amount=('amount', 'sum'))
DUP = dict(company_accounts=int(len(C2)), unique_companies=int(len(g)),
           companies_with_2plus_accounts=int((g.accounts >= 2).sum()), accounts_merged=int(len(C2) - len(g)),
           companies_in_both_dou7_bnn=int((g.cos == 2).sum()))
for co in COS: DUP[f'unique_companies_{co}'] = int(C2[C2.co == co].key.nunique())
display(pd.Series(DUP).to_frame('count').style.format('{:,.0f}'))
print('ตัวอย่างกลุ่มที่รวม (mask):')
rows = []
for gi, key in enumerate(g[g.accounts >= 2].sort_values('lines', ascending=False).index[:6]):
    for _, r in C2[C2.key == key].iterrows(): rows.append((gi + 1, CO_NAME[r.co], mask_org(r['name'])))
pd.DataFrame(rows, columns=['group', 'company', 'name'])

## Top customers (ความเข้มข้นของยอดขาย)
ลูกค้า B2B ไม่กี่รายถือยอดส่วนใหญ่หรือไม่ — นับหลังรวมบัญชีซ้ำแล้ว

In [ ]:
CONC = {}
for co in COS:
    s_ = C2[C2.co == co].groupby('key').amount.sum().sort_values(ascending=False)
    tot_ = A[A.co == co].amount.sum()
    CONC[co] = {'top10_%': float(s_.head(10).sum() / tot_ * 100) if tot_ else 0, 'top50_%': float(s_.head(50).sum() / tot_ * 100) if tot_ else 0}
pd.DataFrame(CONC).style.format('{:.1f}')

## Summary

In [ ]:
KS = {co: {k: dict(accounts=int(v.accounts), lines=int(v.lines), amount=float(v.amount)) for k, v in K.loc[co].iterrows()} for co in COS}
SUMMARY = {'D365': dict(name='D365 Double7 · Adept', accounts={co: int((acct.co == co).sum()) for co in COS}, kinds=KS, dedup=DUP, concentration=CONC,
                        multi_name_accounts=int((names_per_acct > 1).sum())), '_run': time.strftime('%Y-%m-%d %H:%M')}
os.makedirs(OUT, exist_ok=True)
json.dump(SUMMARY, open(os.path.join(OUT, 'd365_cleansing_summary.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=float)
print(json.dumps(SUMMARY['D365']['dedup'], indent=1))

## หมายเหตุ
- ไม่มีเลขผู้เสียภาษีที่ใช้ได้ (`tm_Customer.TaxId` ถูก mask) → รวมด้วยชื่อบริษัทเท่านั้น · ถ้าได้ TaxId จะใช้ TaxId + สาขา (`TaxHeadoffice`) แทน
- ชื่อ "person-like" ของ Adept ควรให้ทีม Adept ช่วยตรวจว่าเป็นบุคคลหรือบริษัท
- ขั้นถัดไป: เทียบบริษัท D365 กับนิติบุคคลใน CRM / ITEC-Rep ด้วย company key